# 01 Data sharing agreement — widget-driven metadata capture


In [ ]:
%run 00_env_config


In [ ]:
from pprint import pprint

from fabricops_kit import (
    collect_agreement_metadata,
    commit_agreement_metadata,
    create_agreement_widgets,
    setup_notebook,
)


## 1. Load config and validate notebook runtime

The `01` notebook captures human-approved agreement metadata only. It does not run profiling, DQ authoring, lineage, or pipeline contract logic.


In [ ]:
env_name = "dev"
notebook_name = "01_da_example_agreement"
commit_to_metadata = False  # Set to True only after steward/business approval.

BOOTSTRAP_01 = setup_notebook(
    config=CONFIG,
    env=env_name,
    required_targets=["metadata"],
    notebook_name=notebook_name,
)

metadata_store = CONFIG.path_config.paths[env_name]["metadata"]


## 2. Prepare optional dropdown lists

Supply project-specific option lists from `00_env_config` when available. If `departments` or `source_systems` are left as `None`, those fields remain free text. Sensitivity labels and refresh frequencies use FabricOps defaults when no custom list is supplied.


In [ ]:
def _config_get(config, key, default=None):
    if isinstance(config, dict):
        return config.get(key, default)
    return getattr(config, key, default)

sensitivity_labels = _config_get(CONFIG, "sensitivity_labels", None)
departments = _config_get(CONFIG, "departments", None)
source_systems = _config_get(CONFIG, "source_systems", None)
refresh_frequencies = _config_get(CONFIG, "refresh_frequencies", None)

create_agreement_widgets(
    sensitivity_labels=sensitivity_labels,
    departments=departments,
    source_systems=source_systems,
    refresh_frequencies=refresh_frequencies,
)

print("Agreement metadata widgets are ready. Fill or confirm the widget values before running the next cell.")


## 3. Collect and preview metadata records

`collect_agreement_metadata` reads widget values, derives `agreement_status` from `expiry_date`, and builds audited records with one shared `committed_at` value.


In [ ]:
lakehouse_name = getattr(metadata_store, "name", str(metadata_store))
runtime_context = {
    "notebook_name": notebook_name,
    "workspace_name": getattr(BOOTSTRAP_01, "workspace_name", ""),
    "lakehouse_name": lakehouse_name,
    "run_id": getattr(BOOTSTRAP_01, "run_id", ""),
}

agreement_metadata = collect_agreement_metadata(runtime_context=runtime_context)
header_record = agreement_metadata["header_record"]
catalogue_record = agreement_metadata["catalogue_record"]
scope_record = agreement_metadata["scope_record"]

print("Agreement metadata summary")
pprint(agreement_metadata["summary"])
print("\nHeader record")
pprint(header_record)
print("\nCatalogue record")
pprint(catalogue_record)
print("\nScope record")
pprint(scope_record)


## 4. Commit agreement metadata

Review the preview before committing. Records are append-friendly and are written to the configured metadata lakehouse target.


In [ ]:
if commit_to_metadata:
    commit_summary = commit_agreement_metadata(
        spark=spark,
        agreement_metadata=agreement_metadata,
        metadata_lakehouse=metadata_store,
        mode="append",
    )
else:
    commit_summary = agreement_metadata["summary"]
    print("Dry run: set commit_to_metadata=True after approval to write agreement metadata.")

print("Agreement metadata commit summary")
for key in [
    "agreement_id",
    "agreement_status",
    "expiry_date",
    "status_as_of_date",
    "committed_by",
    "committed_at",
    "tables_updated",
]:
    print(f"- {key}: {commit_summary[key]}")


## Next step

- `01` anchors agreement-level header, catalogue, and scope metadata under `agreement_id`.
- `02` and `03` can reuse this approved agreement anchor when collecting profiling, DQ, lineage, and pipeline contract evidence.
- Keep later notebook evidence separate from this human-approved agreement capture layer.
